# connections-rl — GRPO v2 on Kaggle 2×T4 with accelerate/FSDP

Settings → Accelerator → **GPU T4 x2**, Internet → On. Kaggle allows ~30 GPU-hours/week.

Designed for **Save & Run All (batch)**: runs detached up to ~12h, checkpoints
every 50 steps, auto-resumes if re-run, and pushes the final adapter to HF Hub
so nothing depends on the session surviving.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_...'   # write token; or use Kaggle Add-ons > Secrets
HF_USER = 'your-hf-username'
os.environ['WANDB_API_KEY'] = '...' # from wandb.ai/authorize
!nvidia-smi

In [ ]:
!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e ".[train]"
# Kaggle/Colab preinstall an old torchao that newer peft rejects; we don't use it.
!pip uninstall -q -y torchao
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local

In [ ]:
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

In [ ]:
# Warm start: SFT adapter from the HF Hub model registry -> artifacts/sft
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft', local_dir='artifacts/sft',
                  token=os.environ['HF_TOKEN'])
!ls artifacts/sft

## Launch GRPO (single GPU — proven path)

FSDP mixed precision fights TRL's `generate()`-based rollouts on T4s
(dtype mismatches during generation). Single-GPU is the battle-tested path;
~10h for 2 epochs, and auto-resume covers the 12h batch limit if needed.
The FSDP config remains in configs/accelerate/ for sm80+ GPUs.

In [ ]:
!python -m connections_rl.train.grpo --config configs/train/grpo.yaml

In [ ]:
# Push the trained adapter to the Hub (model registry). Runs even in batch mode.
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
repo = f'{HF_USER}/connections-rl-grpo'
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path='artifacts/grpo', repo_id=repo,
                  ignore_patterns=['checkpoint-*'])
print('pushed', repo)